In [45]:
import numpy as np
import pandas as pd
import cv2
import os
from pathlib import Path
from collections import Counter

import torch
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device.type}")

Using device: cpu


In [47]:
def check_folder(data_path):
    folder = Path(data_path)
    
    total_files = 0
    total_folders = 0
    total_size = 0
    extensions = []
    
    for item in folder.rglob('*'):  # рекурсивно пройдем по всем вложенным файлам и папкам
        if item.is_file():
            total_files += 1
            total_size += item.stat().st_size
            ext = item.suffix.lower() if item.suffix else 'без расширения'
            extensions.append(ext)
        elif item.is_dir():
            total_folders += 1
    
    ext_counts = Counter(extensions)
    
    folder_stats = (total_files, total_folders, total_size)
    
    print(f"Статистика папки: {data_path}")
    print(f"Всего файлов: {total_files}")
    print(f"Всего папок (включая вложенные): {total_folders}")
    print(f"Общий размер файлов: {total_size / (1024**2):.2f} МБ")
    print("\nРаспределение по расширениям файлов:")
    for ext, count in ext_counts.most_common():
        print(f"  {ext}: {count}")
    return folder_stats

In [48]:
data_path = os.path.join(str(Path.home()), 'PycharmProjects/ML/src/data', 'animals')
if not os.path.exists(data_path):
    os.makedirs(data_path)

в этот раз, для разнообразия, используем для загрузки данных API. Короткая напоминалка, чтобы всё получилось:
1. Создайте (или скачайте) файл `kaggle.json`
	* Перейдите в ваш аккаунт Kaggle: https://www.kaggle.com/
	* Нажмите на ваш профиль → “Account” (Аккаунт)
	* В разделе “API” нажмите “Create New API Token” 
2. Поместите файл `kaggle.json` в нужное место
	* Переместите скачанный файл в папку /Users/aleksandrkoval/.kaggle/
3. Задать нужные права на файл
    * chmod 600 kaggle.json

In [49]:
from kaggle.api.kaggle_api_extended import KaggleApi

folder_stats = check_folder(data_path)
if folder_stats[2] == 0:
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("andrewmvd/animal-faces", path=data_path, unzip=True)
    print("\nЗагрузка завершена")
else:  print("\nПапка не пустая")

Статистика папки: /Users/aleksandrkoval/PycharmProjects/ML/src/data/animals
Всего файлов: 16133
Всего папок (включая вложенные): 9
Общий размер файлов: 695.58 МБ

Распределение по расширениям файлов:
  .jpg: 16130
  без расширения: 3

Папка не пустая


In [56]:
data_train = os.path.join(data_path, 'afhq/train/')
data_val = os.path.join(data_path, 'afhq/val/')

In [82]:
def get_animals_data(classes, data_path):
    allPics = []
    allAnimals = []
    for classElement in classes:
        filenames = os.listdir(os.path.join(data_path, classElement))
        for file in filenames:
            file_name = os.path.join(classElement, file)
            allPics.append(file_name)
        allAnimals.extend([classes[classElement]] * len(filenames))
    
    df = pd.DataFrame({
        'filename': allPics,
        'class': allAnimals
    })
    return df

In [91]:
classes = {"cat": "0", "dog": "1", "wild": "2"}
df = get_animals_data(classes, data_train)
df_test = get_animals_data(classes, data_val)
print("df_test:", df_test.shape) 
display(df_test.head())

df_test: (1500, 2)


,filename,class
0,cat/pixabay_cat_002256.jpg,0
1,cat/pixabay_cat_000441.jpg,0
2,cat/flickr_cat_000802.jpg,0
3,cat/flickr_cat_000816.jpg,0
4,cat/flickr_cat_000180.jpg,0
